# Stanford dragon

The classic scanned model - 871k triangles - loaded straight from the web,
read with VTK and displayed as a K3D mesh. It is a good stress test for the
two physically based renderers: enough geometry to be interesting, and enough
concave detail (the scales, the coiled tail) for occlusion and bounced light to
show.

Additional requirements for this example: `vtk`

Model: [Stanford Computer Graphics Laboratory](https://graphics.stanford.edu/data/3Dscanrep/),
packaged by [Morgan McGuire's Computer Graphics Archive](https://casual-effects.com/data/).

In [ ]:
import zipfile

import numpy as np
import vtk

import k3d
from k3d.helpers import download

## Fetching and reading

The archive is ~18 MB and holds a single `.obj`. VTK reads it; a triangle
filter guarantees triangles, since K3D meshes are triangle soups and an OBJ
may legally contain polygons. The scan is authored Y-up while K3D works Z-up,
so it is rotated once here - doing it in VTK rather than through the mesh's
model matrix keeps the bounds honest for the camera and floor below.

In [ ]:
filename = download('https://casual-effects.com/g3d/data10/research/model/dragon/dragon.zip')

with zipfile.ZipFile(filename) as archive:
    archive.extract('dragon.obj')

reader = vtk.vtkOBJReader()
reader.SetFileName('dragon.obj')

triangles = vtk.vtkTriangleFilter()
triangles.SetInputConnection(reader.GetOutputPort())

to_z_up = vtk.vtkTransform()
to_z_up.RotateX(90)

upright = vtk.vtkTransformPolyDataFilter()
upright.SetTransform(to_z_up)
upright.SetInputConnection(triangles.GetOutputPort())
upright.Update()

dragon = upright.GetOutput()
dragon.GetNumberOfPoints(), dragon.GetNumberOfCells()

## Building the plot

`vtk_poly_data` turns the polydata into a K3D mesh. The model spans about one
unit, sits off-centre, and its normals come from the scan - `flat_shading=False`
keeps the surface smooth.

`compression_level` matters here: the vertex buffer is 30 MB, and compressing it
keeps the notebook (and any HTML snapshot of it) manageable.

In [ ]:
bounds = np.array(dragon.GetBounds()).reshape(3, 2)
centre = bounds.mean(axis=1)
size = float((bounds[:, 1] - bounds[:, 0]).max())

mesh = k3d.vtk_poly_data(dragon,
                         color=0xC8A15A,
                         flat_shading=False,
                         roughness=0.2,
                         metalness=0.7,
                         compression_level=5,
                         name='dragon')

plot = k3d.plot(renderer='advanced',
                environment='studio',
                grid_visible=False,
                camera_auto_fit=False)
plot += mesh

# three-quarter view from slightly above, framed on the model's own size
# the head points along XY (0.75, 0.66): standing 28 degrees off that axis keeps it
# turned towards the camera while the arc of the back and the coiled tail stay visible
eye = centre + np.array([1.02, 0.245, 0.38]) * size
plot.camera = [*eye, *centre, 0, 0, 1]

plot.display()

## Path tracing the same scene

`cinematic` traces the light instead of approximating it: the shadow under the
belly, the light that bounces off the floor into the scales, and the sheen along
the spine all fall out of the simulation. The image refines sample by sample and
the counter in the corner tells you where it is; every camera move starts it over.

A floor gives the bounced light something to come from - without it a path traced
model floats in the environment and looks flatter than it should.

In [ ]:
floor_z = float(bounds[2, 0])
span = 0.9 * size

plot += k3d.mesh(np.array([[centre[0] - span, centre[1] - span, floor_z],
                           [centre[0] + span, centre[1] - span, floor_z],
                           [centre[0] + span, centre[1] + span, floor_z],
                           [centre[0] - span, centre[1] + span, floor_z]], np.float32),
                 np.array([[0, 1, 2], [0, 2, 3]], np.uint32),
                 color=0x9AA0A6,
                 roughness=0.9,
                 name='floor')

plot.renderer = 'cinematic'
plot.cinematic_samples = 96
plot.cinematic_bounces = 6

Bright bounced light between surfaces exceeds what a display can show, so a
tone curve is worth having:

In [ ]:
plot.tone_mapping = 'aces'

Raise the budget for a final image - it is a hard ceiling, and the loop parks
itself once it is reached:

In [ ]:
plot.cinematic_samples = 512